# 02 — Silver Transactions Cleaning

**Purpose:** Convert raw Bronze events into trusted, standardized, analytics-ready event records while preserving invalid records in quarantine.

**Flow:** `Bronze Delta → standardize → validate → quality flags → clean / quarantine`

Silver remains at event-level grain so the cleaned data can support multiple downstream Gold products.


In [ ]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lower,
    to_date,
    trim,
    when,
)

BRONZE_TABLE = "ecommerce_lakehouse.bronze.transactions_raw"
SILVER_CLEAN_TABLE = "ecommerce_lakehouse.silver.transactions_clean"
SILVER_QUARANTINE_TABLE = "ecommerce_lakehouse.silver.transactions_quarantine"

SILVER_VALID_CHECKPOINT = (
    "/Volumes/ecommerce_lakehouse/raw/pipeline_metadata/"
    "silver_valid_checkpoint"
)
SILVER_QUARANTINE_CHECKPOINT = (
    "/Volumes/ecommerce_lakehouse/raw/pipeline_metadata/"
    "silver_quarantine_checkpoint"
)

VALID_EVENT_TYPES = [
    "view",
    "cart",
    "remove_from_cart",
    "purchase",
]


## Read Bronze incrementally

Bronze is already a Delta table, so Auto Loader is no longer required. Structured Streaming consumes newly committed Bronze data using Delta as the source.


In [ ]:
bronze_stream = (
    spark.readStream
        .table(BRONZE_TABLE)
)


## Standardize source fields

String values are normalized before validation, and `_silver_processed_at` records when each event was transformed into Silver.


In [ ]:
silver_candidate = (
    bronze_stream
        .withColumn(
            "event_type",
            lower(trim(col("event_type")))
        )
        .withColumn(
            "brand",
            lower(trim(col("brand")))
        )
        .withColumn(
            "_silver_processed_at",
            current_timestamp()
        )
)


## Apply business-aware validation

A row is quarantined when a required business field is missing, the event type is unsupported, the price is negative, or Auto Loader rescued schema-mismatched data.

A **zero price is retained** rather than rejected. Profiling showed zero-price rows were overwhelmingly behavioral events, so `_price_quality = "ZERO_PRICE"` preserves those events for behavioral analytics while allowing monetary Gold metrics to use only trusted prices.


In [ ]:
validated_df = (
    silver_candidate
        .withColumn(
            "_is_valid",
            when(col("event_time").isNull(), False)
            .when(col("event_type").isNull(), False)
            .when(~col("event_type").isin(VALID_EVENT_TYPES), False)
            .when(col("product_id").isNull(), False)
            .when(col("user_id").isNull(), False)
            .when(col("price").isNull(), False)
            .when(col("price") < 0, False)
            .when(col("_rescued_data").isNotNull(), False)
            .otherwise(True)
        )
        .withColumn(
            "_validation_error",
            when(col("event_time").isNull(), "NULL_EVENT_TIME")
            .when(col("event_type").isNull(), "NULL_EVENT_TYPE")
            .when(~col("event_type").isin(VALID_EVENT_TYPES), "INVALID_EVENT_TYPE")
            .when(col("product_id").isNull(), "NULL_PRODUCT_ID")
            .when(col("user_id").isNull(), "NULL_USER_ID")
            .when(col("price").isNull(), "NULL_PRICE")
            .when(col("price") < 0, "INVALID_PRICE")
            .when(col("_rescued_data").isNotNull(), "RESCUED_DATA")
        )
        .withColumn(
            "_price_quality",
            when(col("price") == 0, "ZERO_PRICE")
            .otherwise("VALID")
        )
)


## Split clean and quarantined events

The clean output drops temporary validation fields and adds `event_date` for downstream date-level processing. Invalid rows retain the validation reason for investigation.


In [ ]:
valid_df = (
    validated_df
        .filter(col("_is_valid") == True)
        .drop("_is_valid", "_validation_error")
        .withColumn("event_date", to_date(col("event_time")))
)

quarantine_df = (
    validated_df
        .filter(col("_is_valid") == False)
)


## Persist both Silver outputs

The clean and quarantine tables are separate streaming sinks, so each uses its own checkpoint.


In [ ]:
silver_query = (
    valid_df.writeStream
        .format("delta")
        .option("checkpointLocation", SILVER_VALID_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(SILVER_CLEAN_TABLE)
)

silver_query.awaitTermination()


In [ ]:
quarantine_query = (
    quarantine_df.writeStream
        .format("delta")
        .option("checkpointLocation", SILVER_QUARANTINE_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(SILVER_QUARANTINE_TABLE)
)

quarantine_query.awaitTermination()


### Duplicate-handling decision

The source does not provide a trustworthy unique `event_id`. Identical-looking business rows may therefore represent either source duplicates or legitimate repeated user actions. The pipeline intentionally avoids blind `dropDuplicates()` logic to prevent deleting potentially valid events.
